## 4. Validación de Hipótesis de la Cátedra
Carga de librerías y lectura del dataset limpio proporcionado por el equipo.

In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm

df = pd.read_parquet('dataset_limpio.parquet')
df.head()

,product_name,brands,categories,ingredients,nova_group,allergens,energy_kcal,fat_100g,saturated_fat_100g,carbs_100g,...,sodium_100g,contains_gluten,contains_dairy,contains_nuts,contains_soy,contains_eggs,contains_fish,nutriscore_num,ecoscore_num,categoria_macro
0,Sidi Ali,سيدي علي,"en:beverages-and-beverages-preparations, en:be...",OBD1 999 999 1112606 266963207 mb,<NA>,NaN,0.0,0.0,0.0,4.2,...,0.000000,False,False,False,False,False,False,5,<NA>,Bebidas
1,Perly,Perly,"en:dairies, en:fermented-foods, en:fermented-m...","milk cream, cream, sugar, banana, bacteria",3,"en:banana, en:milk",97.0,3.0,0.0,9.4,...,0.000000,False,True,False,False,False,False,<NA>,4,Lácteos
2,اكوافينا,AQUAFINA,"en:beverages-and-beverages-preparations, en:be...",ouverture et avant le : Voir bouteille. après ...,<NA>,NaN,0.0,0.0,0.0,0.0,...,0.000203,False,False,False,False,False,False,5,<NA>,Bebidas
3,Lait,Jaouda,"en:dairies, en:meals, en:milks-liquid-and-powder",Lait frais pasteurisé demi-écrémé à 15g/l de m...,3,en:milk,45.0,1.5,1.0,4.9,...,0.090000,False,True,False,False,False,False,4,2,Lácteos
4,Eau De Source,Cristaline,"en:beverages-and-beverages-preparations, en:be...",Eau de source,1,en:none,0.0,0.0,0.0,0.0,...,0.001100,False,False,False,False,False,False,5,<NA>,Bebidas


### 4.c Modelo probabilístico de calificación Nutri-Score
**Hipótesis:** El grado de procesamiento industrial (NOVA), la categoría del producto y la cantidad de alérgenos declarados explican conjuntamente la probabilidad de que un producto reciba una calificación Nutri-Score desfavorable (D/E), sin utilizar directamente su composición nutricional.

* **Variable Dependiente (Y):** Probabilidad de Nutri-Score desfavorable (1 si es D o E; 0 si es A, B o C).
* **Variables Independientes (X):** `nova_group` (cuantitativa discreta), `cant_alergenos` (cuantitativa discreta derivada) y `categories` (cualitativa nominal, procesada mediante variables dummy).
* **Prueba estadística:** Regresión Logística Binaria Múltiple. Evaluaremos la significancia global del modelo (LLR p-value) y los coeficientes individuales de cada variable predictora.

**Nota metodológica:** Para abordar esta hipótesis, elegimos utilizar una Regresión Logística Binaria Múltiple. Dado que el objetivo es modelar la *probabilidad* de un resultado dicotómico (Nutri-Score Desfavorable = 1, Favorable = 0), una regresión lineal tradicional no es viable porque podría arrojar probabilidades ilógicas fuera del rango [0, 1]. Además, a diferencia de modelos puramente predictivos (como Random Forest o Árboles de Decisión), la regresión logística nos proporciona los p-valores exactos de cada coeficiente. Esto es indispensable para realizar inferencia estadística y validar matemáticamente la significancia de cada variable sobre la hipótesis planteada.

In [6]:
if 'allergens_n' not in df.columns and 'allergens' in df.columns:
    # Cuenta los alérgenos separados por coma ignorando valores nulos
    df['allergens_n'] = df['allergens'].fillna('').apply(
        lambda x: len([a for a in x.split(',') if a.strip()])
    )

resumen_alergenos = df.groupby('categoria_macro')['allergens_n'].agg(
    total_alergenos='sum',
    promedio_por_producto='mean',
    productos_con_alergenos=lambda x: (x > 0).sum()
).sort_values(by='total_alergenos', ascending=False)

print("--- RESUMEN DE ALÉRGENOS POR CATEGORÍA MACRO ---")
print(resumen_alergenos)

--- RESUMEN DE ALÉRGENOS POR CATEGORÍA MACRO ---
                        total_alergenos  promedio_por_producto  \
categoria_macro                                                  
Snacks y Dulces                    2273               1.954428   
Vegetales y Cereales               1694               1.130087   
Lácteos                             639               0.950893   
Grasas y Salsas                     333               1.295720   
Bebidas                             284               0.457327   
Comidas Preparadas                  269               1.393782   
Carnes y Pescados                   194               1.469697   
Otros                                61               0.610000   
Untables                             24               1.090909   
Caldos y Sopas                       16               1.142857   
Suplementos y Aditivos                5               0.384615   

                        productos_con_alergenos  
categoria_macro                           

#### vemos que tenemos muchos alérgenos, probablemente provocado por la alta cantidad de idiomas (lo cual puede ocasionar que variables como milk y leche sumen 1 a la cantidad total). Sin embargo, hacer esta limpieza de forma correcta resultaría muy costosa, por lo que de todos modos abordaremos el problema, pero teniendo esto en cuenta

In [7]:

df['nutriscore_malo'] = (df['nutriscore_num'] <= 2).astype(float)

columnas_modelo = ['nova_group', 'allergens_n', 'categoria_macro', 'nutriscore_malo']
df_4c = df[columnas_modelo].dropna().copy()

# Codificación One-Hot para categoria_macro (drop_first=True evita multicolinealidad)
X_cat = pd.get_dummies(df_4c['categoria_macro'], drop_first=True, dtype=float)

# Definir predictores y variable objetivo
X = df_4c[['nova_group', 'allergens_n']].astype(float).join(X_cat)
Y = df_4c['nutriscore_malo']

X = sm.add_constant(X)

modelo_logistico = sm.Logit(Y, X).fit()

print(modelo_logistico.summary())

         Current function value: 0.522155
         Iterations: 35
                           Logit Regression Results                           
Dep. Variable:        nutriscore_malo   No. Observations:                 4101
Model:                          Logit   Df Residuals:                     4088
Method:                           MLE   Df Model:                           12
Date:                Tue, 10 Nov 2026   Pseudo R-squ.:                  0.2349
Time:                        16:22:10   Log-Likelihood:                -2141.4
converged:                      False   LL-Null:                       -2798.7
Covariance Type:            nonrobust   LLR p-value:                3.434e-274
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     -1.9739      0.176    -11.222      0.000      -2.319      -1.629
nova_group                 0.

/home/ezequiel/Escritorio/fund_cienc_datos/TP-Fundamentos-de-la-ciencia-de-datos/.venv/lib/python3.12/site-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


### Para este primer modelo se obtuvieron resultados favorables en términos de significancia global, ya que el LLR p-value indica que el modelo mejora significativamente respecto del modelo nulo. Sin embargo, al analizar la significancia individual de las variables, observamos que algunas categorías, como suplementos y aditivos, caldos y sopas y lácteos, no presentan evidencia estadística suficiente para afirmar que tienen una asociación significativa con la variable dependiente, manteniendo constantes las demás variables.

Una posible explicación es que estas categorías no aporten información adicional relevante al modelo o que su efecto no pueda distinguirse claramente del de otras variables incluidas.

Por este motivo, planteamos evaluar un segundo modelo excluyendo la categoría de suplementos y aditivos, para analizar si su eliminación produce una mejora en el ajuste general y modifica la significancia de las variables restantes.

In [8]:
# Agrupar 'Suplementos y Aditivos' dentro de 'Otros'
df_4c['categoria_macro'] = df_4c['categoria_macro'].replace({'Suplementos y Aditivos': 'Otros'})

X_cat = pd.get_dummies(df_4c['categoria_macro'], drop_first=True, dtype=float)
X = df_4c[['nova_group', 'allergens_n']].astype(float).join(X_cat)
X = sm.add_constant(X)

modelo_logistico_2 = sm.Logit(Y, X).fit()
print(modelo_logistico_2.summary())

Optimization terminated successfully.
         Current function value: 0.522589
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:        nutriscore_malo   No. Observations:                 4101
Model:                          Logit   Df Residuals:                     4089
Method:                           MLE   Df Model:                           11
Date:                Tue, 10 Nov 2026   Pseudo R-squ.:                  0.2342
Time:                        16:22:10   Log-Likelihood:                -2143.1
converged:                       True   LL-Null:                       -2798.7
Covariance Type:            nonrobust   LLR p-value:                1.799e-274
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   -1.9662      0.176    -11.195      0.000      -2.310      -1.622

In [9]:
comparacion = pd.DataFrame({
    'Indicador': ['Pseudo R²', 'LLR p-value', 'AIC', 'BIC'],
    'Modelo 1': [
        modelo_logistico.prsquared,
        modelo_logistico.llr_pvalue,
        modelo_logistico.aic,
        modelo_logistico.bic
    ],
    'Modelo 2': [
        modelo_logistico_2.prsquared,
        modelo_logistico_2.llr_pvalue,
        modelo_logistico_2.aic,
        modelo_logistico_2.bic
    ]
})

display(comparacion)

,Indicador,Modelo 1,Modelo 2
0,Pseudo R²,2.348723e-01,2.342361e-01
1,LLR p-value,3.433687e-274,1.798521e-274
2,AIC,4.308717e+03,4.310278e+03
3,BIC,4.390864e+03,4.386106e+03


### Si bien el modelo mejora un poco su parsimonia manteniendo el poder explicativo, los cambios no son muy notables. Veamos que pasa al incluir los caldos y sopas y los lacteos en el conjunto de bebidas

In [12]:

df_4c['categoria_macro'] = df_4c['categoria_macro'].replace({
    'Caldos y Sopas': 'Bebidas',
    'Lácteos': 'Bebidas'
})

X_cat = pd.get_dummies(df_4c['categoria_macro'], drop_first=True, dtype=float)

X = df_4c[['nova_group', 'allergens_n']].astype(float).join(X_cat)
X = sm.add_constant(X)
Y = df_4c['nutriscore_malo']

modelo_logistico_3 = sm.Logit(Y, X).fit()
print(modelo_logistico_3.summary())

Optimization terminated successfully.
         Current function value: 0.522720
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:        nutriscore_malo   No. Observations:                 4101
Model:                          Logit   Df Residuals:                     4091
Method:                           MLE   Df Model:                            9
Date:                Tue, 10 Nov 2026   Pseudo R-squ.:                  0.2340
Time:                        17:29:44   Log-Likelihood:                -2143.7
converged:                       True   LL-Null:                       -2798.7
Covariance Type:            nonrobust   LLR p-value:                2.104e-276
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   -1.8816      0.154    -12.228      0.000      -2.183      -1.580

In [ ]:

df_4c['categoria_macro'] = df_4c['categoria_macro'].replace({
    'Caldos y Sopas': 'Bebidas',
    'Lácteos': 'Bebidas'
})

X_cat = pd.get_dummies(df_4c['categoria_macro'], drop_first=True, dtype=float)

X = df_4c[['nova_group', 'allergens_n']].astype(float).join(X_cat)
X = sm.add_constant(X)
Y = df_4c['nutriscore_malo']

modelo_logistico_3 = sm.Logit(Y, X).fit()
print(modelo_logistico_3.summary())

Optimization terminated successfully.
         Current function value: 0.522720
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:        nutriscore_malo   No. Observations:                 4101
Model:                          Logit   Df Residuals:                     4091
Method:                           MLE   Df Model:                            9
Date:                Tue, 10 Nov 2026   Pseudo R-squ.:                  0.2340
Time:                        17:29:44   Log-Likelihood:                -2143.7
converged:                       True   LL-Null:                       -2798.7
Covariance Type:            nonrobust   LLR p-value:                2.104e-276
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   -1.8816      0.154    -12.228      0.000      -2.183      -1.580

In [11]:
comparacion = pd.DataFrame({
    'Indicador': ['Pseudo R²', 'LLR p-value', 'AIC', 'BIC'],
    'Modelo 2': [
        modelo_logistico_2.prsquared,
        modelo_logistico_2.llr_pvalue,
        modelo_logistico_2.aic,
        modelo_logistico_2.bic
    ],
    'Modelo 3': [
        modelo_logistico_3.prsquared,
        modelo_logistico_3.llr_pvalue,
        modelo_logistico_3.aic,
        modelo_logistico_3.bic
    ]
})

display(comparacion)

,Indicador,Modelo 2,Modelo 3
0,Pseudo R²,2.342361e-01,2.340440e-01
1,LLR p-value,1.798521e-274,2.103958e-276
2,AIC,4.310278e+03,4.307353e+03
3,BIC,4.386106e+03,4.370543e+03


### Vemos que el modelo 3 es claramente mejor al modelo 2, ya que el AIC y el BIC se reducieron con una diferencia 15,56 y 2,93 puntos (confirmando mayor eficiencia y evidencia estadística para este modelo)

### Conclusión de la Hipótesis 4.c

Los resultados de la regresión logística confirman la hipótesis planteada: es posible explicar la probabilidad de que un producto reciba una calificación Nutri-Score desfavorable (D o E) a partir de su grado de procesamiento industrial, su categoría y la presencia de alérgenos, sin recurrir a variables de composición nutricional directa.

En términos del ajuste global, el modelo resulta altamente significativo ($p\text{-valor LLR} = 2.104 \times 10^{-276}$) y alcanza un Pseudo $R^2$ de McFadden de $0.2340$. En regresión logística, un Pseudo $R^2$ superior a $0.20$ refleja un poder explicativo muy sólido sobre los datos.

Al analizar la contribución de cada predictor:

* **Grado de procesamiento (nova_group):** Muestra un impacto positivo y estadísticamente significativo ($\beta = 0.4719, p < 0.001$). En términos de *Odds Ratio*, por cada salto en la clasificación NOVA, la probabilidad a favor de obtener un Nutri-Score malo se incrementa en un $60.3\%$ ($\text{OR} = 1.603$).
* **Cantidad de alérgenos (allergens_n):** Resulta ser un predictor significativo dentro del modelo ($\beta = 0.0881, p = 0.008$), indicando que la presencia de cada alérgeno único adicional aumenta el riesgo de una calificación desfavorable en un $9.2\%$ ($\text{OR} = 1.092$).
* **Categorías de alimentos (frente a la línea base unificada de Bebidas, Lácteos y Caldos):**
* **Categorías de mayor riesgo:** Formar parte de **Otros** ($\text{OR} = 4.31$), **Snacks y Dulces** ($\text{OR} = 3.36$), **Untables** ($\text{OR} = 3.35$) o **Grasas y Salsas** ($\text{OR} = 2.71$) multiplica de manera significativa la probabilidad de recibir letras D o E.
* **Categorías protectoras:** Las familias de **Comidas Preparadas** ($\text{OR} = 0.21$), **Carnes y Pescados** ($\text{OR} = 0.24$) y **Vegetales y Cereales** ($\text{OR} = 0.24$) disminuyen sustancialmente la probabilidad de obtener un perfil Nutri-Score negativo frente al grupo de referencia.